In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(r"C:\Project 3(Advanced Multi-Source Data Pipeline)\1.1_crypto_data.csv")
df.head()

,id,symbol,name,image,current_price,market_cap,market_cap_rank,fully_diluted_valuation,total_volume,high_24h,...,total_supply,max_supply,ath,ath_change_percentage,ath_date,atl,atl_change_percentage,atl_date,roi,last_updated
0,bitcoin,btc,Bitcoin,https://coin-images.coingecko.com/coins/images...,78364.000000,1573174135422,1,1573176877328,2.804003e+10,79625.000000,...,2.008136e+07,2.100000e+07,126080.00,-37.84587,2025-10-06T10:57:42.000Z,67.810000,1.154656e+05,2013-07-05T16:00:00.000Z,NaN,2026-09-08T09:09:20.000Z
1,ethereum,eth,Ethereum,https://coin-images.coingecko.com/coins/images...,2476.390000,302087597694,2,302087597694,1.078214e+10,2510.030000,...,1.220297e+08,NaN,4946.05,-49.93190,2025-08-24T11:21:03.000Z,0.432979,5.718430e+05,2015-10-19T16:00:00.000Z,"{'times': 41.25187165701799, 'currency': 'btc'...",2026-09-08T09:09:20.000Z
2,tether,usdt,Tether,https://coin-images.coingecko.com/coins/images...,0.999775,183397423736,3,188863142989,5.217489e+10,0.999963,...,1.889055e+11,NaN,1.32,-24.43664,2018-07-23T16:00:00.000Z,0.572521,7.462682e+01,2015-03-01T16:00:00.000Z,NaN,2026-09-08T09:09:20.000Z
3,binancecoin,bnb,BNB,https://coin-images.coingecko.com/coins/images...,752.400000,100144405327,4,100144399025,1.087073e+09,760.330000,...,1.331615e+08,2.000000e+08,1369.99,-45.07986,2025-10-13T00:41:24.000Z,0.039818,1.889515e+06,2017-10-18T16:00:00.000Z,NaN,2026-09-08T09:09:20.000Z
4,ripple,xrp,XRP,https://coin-images.coingecko.com/coins/images...,1.390000,87168896250,5,138906774820,1.895227e+09,1.410000,...,9.998563e+10,1.000000e+11,3.65,-61.88953,2025-07-17T19:40:53.000Z,0.002686,5.163287e+04,2014-05-21T16:00:00.000Z,NaN,2026-09-08T09:09:20.000Z


In [3]:
# Cleaning missing Values
print(df.isnull().sum())

id                                   0
symbol                               0
name                                 0
image                                0
current_price                        0
market_cap                           0
market_cap_rank                      0
fully_diluted_valuation              0
total_volume                         0
high_24h                             0
low_24h                              0
price_change_24h                     1
price_change_percentage_24h          1
market_cap_change_24h                1
market_cap_change_percentage_24h     1
circulating_supply                   0
total_supply                         0
max_supply                          49
ath                                  0
ath_change_percentage                0
ath_date                             0
atl                                  0
atl_change_percentage                0
atl_date                             0
roi                                 90
last_updated             

In [4]:
columns = ["current_price","price_change_24h", "market_cap", "total_volume","max_supply",
           "price_change_percentage_24h","market_cap_change_24h","market_cap_change_percentage_24h"]

for col in columns:
    df[col] = df[col].fillna(df[col].median())
    
    df["roi"] = df["roi"].fillna("Unknown")


print(df.isnull().sum())

id                                  0
symbol                              0
name                                0
image                               0
current_price                       0
market_cap                          0
market_cap_rank                     0
fully_diluted_valuation             0
total_volume                        0
high_24h                            0
low_24h                             0
price_change_24h                    0
price_change_percentage_24h         0
market_cap_change_24h               0
market_cap_change_percentage_24h    0
circulating_supply                  0
total_supply                        0
max_supply                          0
ath                                 0
ath_change_percentage               0
ath_date                            0
atl                                 0
atl_change_percentage               0
atl_date                            0
roi                                 0
last_updated                        0
dtype: int64

In [5]:
# convert numerical columns to float
print(df.dtypes)

id                                      str
symbol                                  str
name                                    str
image                                   str
current_price                       float64
market_cap                            int64
market_cap_rank                       int64
fully_diluted_valuation               int64
total_volume                        float64
high_24h                            float64
low_24h                             float64
price_change_24h                    float64
price_change_percentage_24h         float64
market_cap_change_24h               float64
market_cap_change_percentage_24h    float64
circulating_supply                  float64
total_supply                        float64
max_supply                          float64
ath                                 float64
ath_change_percentage               float64
ath_date                                str
atl                                 float64
atl_change_percentage           

In [6]:
df[["market_cap_rank", "market_cap", "fully_diluted_valuation"]] = df[
    ["market_cap_rank", "market_cap", "fully_diluted_valuation"]].astype(float)

print(df.dtypes)

id                                      str
symbol                                  str
name                                    str
image                                   str
current_price                       float64
market_cap                          float64
market_cap_rank                     float64
fully_diluted_valuation             float64
total_volume                        float64
high_24h                            float64
low_24h                             float64
price_change_24h                    float64
price_change_percentage_24h         float64
market_cap_change_24h               float64
market_cap_change_percentage_24h    float64
circulating_supply                  float64
total_supply                        float64
max_supply                          float64
ath                                 float64
ath_change_percentage               float64
ath_date                                str
atl                                 float64
atl_change_percentage           

In [7]:
# Extreme outliers Detecting using IQR + Isolation Forest

columns = ["current_price", "market_cap", "total_volume"]

for col in columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    print(col)
    print(df[(df[col] < lower) | (df[col] > upper)])
    


current_price
                    id symbol                name  \
0              bitcoin    btc             Bitcoin   
1             ethereum    eth            Ethereum   
3          binancecoin    bnb                 BNB   
6               solana    sol              Solana   
9                zcash    zec               Zcash   
10         hyperliquid   hype         Hyperliquid   
14              monero    xmr              Monero   
16            whitebit    wbt       WhiteBIT Coin   
20        bitcoin-cash    bch        Bitcoin Cash   
24            litecoin    ltc            Litecoin   
37         tether-gold   xaut         Tether Gold   
41           bittensor    tao           Bittensor   
42                 okb    okb                 OKB   
46                aave   aave                Aave   
47            pax-gold   paxg            PAX Gold   
73  blockchain-capital   bcap  Blockchain Capital   
74       quant-network    qnt               Quant   
80        venice-token    vvv   

In [11]:
# Inconsistent nulls max_supply, total_supply handling

df["max_supply"] = df["max_supply"].fillna(0)
df["total_supply"] = df["total_supply"].fillna(0)

print(df[["max_supply", "total_supply"]].isnull().sum())

max_supply      0
total_supply    0
dtype: int64


In [12]:
# Convert last_updated to datetime
df["last_updated"] = pd.to_datetime(df["last_updated"], errors="coerce")

# Remove duplicate coins
df = df.drop_duplicates(subset="id")

print(df["last_updated"].dtype)
print(df.duplicated(subset="id").sum())

datetime64[us, UTC]
0


In [14]:
# Remove invalid prices
df = df[df["current_price"] > 0]
print(df["current_price"].min())

# Handle missing percentage changes
df["price_change_percentage_24h"] = (
    df["price_change_percentage_24h"].fillna(0)
)

print(df["price_change_percentage_24h"])

1.69e-06
0     -1.46872
1     -0.74224
2     -0.01227
3      1.01562
4     -1.18714
        ...   
95    16.50414
96    -0.87221
97    -0.51237
98    -0.46225
99    -1.35323
Name: price_change_percentage_24h, Length: 100, dtype: float64


In [15]:
print(df["price_change_percentage_24h"].isnull().sum())

0


In [16]:
# Market Cap Category
df["market_cap_category"] = pd.cut(
    df["market_cap"],
    bins=[0, 1e9, 10e9, float("inf")],
    labels=["Small", "Medium", "Large"]
)

# Volatility Score
df["volatility_score"] = df["price_change_percentage_24h"].abs()

# Supply Ratio
df["supply_ratio"] = df["circulating_supply"] / df["total_supply"]

In [17]:
df.to_csv("crypto_cleaned.csv", index=False)